# Visualize your pexl project

Create a schematic building section from a PEExcel scenario, arrange its buildings,
and illustrate their uses and building services. The same layout can be rendered as
an architectural section, a hand-drawn sketch or a tinted blueprint.

The examples start with an imported project, then use a small teaching project to
explore layout and HVAC choices. All drawings use `pexl.spatial`; no separate symbol
gallery or notebook helper module is needed. Parameters are edited in ordinary Python
cells. The figures are static SVGs.

A section is an area-preserving projection of `scenario.v`. Its depths, building
arrangement and equipment locations are schematic choices, not a surveyed model or
an additional energy calculation. Scientific inputs remain in the scenario.

In [ ]:
from base64 import b64encode
from copy import deepcopy
from pathlib import Path
from xml.etree import ElementTree as ET
import json
import math

import pandas as pd
from IPython.display import HTML, display

from pexl import Project
from pexl.spatial import (
    normalize_layout,
    preset_visuals,
    render_svg,
    section_layout_from_scenario,
    suggest_section_layout,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def show_section(svg):
    """Display a responsive, isolated SVG image in a notebook output."""
    ET.fromstring(svg)
    encoded = b64encode(svg.encode("utf-8")).decode("ascii")
    display(HTML(f'<img alt="Schematic building section" style="max-width:100%;height:auto" '
                 f'src="data:image/svg+xml;base64,{encoded}">'))

## 1. Read a project and draw its first section

Load the project export used in the Excel how-to. We select the Forsthausgasse baseline
by its stored scenario name. The drawing uses two buildings with a 2:1 length ratio;
15 m depth and four above-ground floors are explicit presentation assumptions.
The scenario's recorded building count is retained in the source data.

In [ ]:
project = Project.from_excel(ROOT / "data/exports/ka_project_backup_v1_13.xlsx")
baseline = next(s for s in project.scenarios if s.column_name == "Forsthausgasse | BT2 KON FW")

real_layout = {
    "layout_version": 1,
    "site": {"gap_m": 20},
    "defaults": {
        "geometry": {"depth_m": 15, "max_floors": 4},
        "allocation": {"usages": ["residential"]},
    },
    "buildings": [
        {"id": "A", "name": "Forsthausgasse A", "geometry": {"relative_length": 2}},
        {"id": "B", "name": "Forsthausgasse B", "geometry": {"relative_length": 1}},
    ],
}
real_section = section_layout_from_scenario(baseline, layout=real_layout)
real_svg = render_svg(real_section)
show_section(real_svg)

The section contains geometry and provenance as ordinary dictionaries. Inspecting a
few source fields helps distinguish recorded inputs from layout choices. In this
export, an active thermal slot indicates district heating, but its building location
is unknown. Unlocated equipment is listed rather than placed in an invented room.

In [ ]:
fields = ["NFA_total", "NFA_residential", "building_count", "storey_count_avg",
          "hull_fenestration_rate", "PV_module_area", "PV_kWp",
          "QHmax_1el_m2", "QHmax_2th_m2", "heat_th2_carrier_type"]
display(pd.Series({name: getattr(baseline.v, name, None) for name in fields}, name="Stored input"))
display(pd.DataFrame(real_section.get("equipment", [])))
print(real_svg[:240] + " …")

## 2. Create a teaching project with every usage

This example specifies all seven usage areas, five buildings and an average of 5.5
storeys. The generator reads existing schema variables. Here `NFA_total` is explicitly
set to the sum of the usage areas; no Excel recalculation is required to draw a section.

In [ ]:
demo_project = Project()
scenario = demo_project.get_or_create_scenario("Teaching district | Mixed uses")
v = scenario.v
v.project_name = "Teaching district"
v.project_scenario_name = "Mixed uses"

usage_areas = {
    "NFA_office": 1350.0,
    "NFA_retailfood": 600.0,
    "NFA_retailother": 450.0,
    "NFA_schoolprim": 600.0,
    "NFA_schoolsec": 2000.0,
    "NFA_other": 450.0,
    "NFA_residential": 5000.0,
}
for name, area in usage_areas.items():
    setattr(v, name, area)
v.NFA_total = sum(usage_areas.values())
v.building_count = 5
v.storey_count_avg = 5.5
v.rh_avg = 3.0
v.rh_ceiling = 0.35
v.hull_fenestration_rate = 0.30
v.hull_transmittance_walls = 0.30
v.hull_transmittance_roof = 0.20
v.PV_is_used = 1
v.PV_m2_per_kWp = 5.0
v.PV_kWp = 120.0
v.PV_module_area = None  # derive module area from capacity and m²/kWp

default_section = section_layout_from_scenario(scenario, seed=42, gap_m=20)
show_section(render_svg(default_section))

The default layout favours continuous usage blocks, ground-floor retail and schools,
office between the lower uses and residential, and residential upper floors. It uses
the available `other` and residential budgets to complete floors. The two alternatives
are `separateResidential` and `equalspread`. Change the cell below to compare them.

In [ ]:
layout_mode = "equalspread"  # "default", "separateResidential" or "equalspread"
alternative = section_layout_from_scenario(scenario, layout=layout_mode, seed=42)
show_section(render_svg(alternative))

## 3. Arrange buildings, storeys and basements

A layout dictionary gives more control than a named heuristic. `buildings` determines
the building count; `allocation.usages` lists uses from bottom to top. Relative length,
depth and maximum floors determine capacity. Shared usages follow that capacity unless
`usage_shares` specifies fractions of the scenario-wide usage budget. The generator
derives lengths to fit the areas; the last storey may be partial.

The axes restart at each building. The length scale is continuous, while the second
axis reports floor area per storey: 15 m length means 225 m² at 15 m depth, or 300 m²
at 20 m depth. The gap is presentation space and contributes no NFA.

In [ ]:
layout = {
    "layout_version": 1,
    "site": {"gap_m": 20, "gaps": [{"between": ["A", "B"], "width_m": 25, "kind": "courtyard"}]},
    "defaults": {
        "geometry": {"depth_m": 15, "max_floors": 5, "max_basements": 1},
        "allocation": {"basement_usages": ["other"]},
        "systems": {"plant": [], "roof": ["pv"], "usages": {"*": {"ventilation": "none"}}},
    },
    "buildings": [
        {"id": "A", "name": "Learning and work",
         "geometry": {"relative_length": 2, "max_floors": 4},
         "allocation": {"usages": ["retailfood", "retailother", "schoolprim", "schoolsec", "office", "residential"]},
         "envelope": {"street": {"facade": {"style": "gründerzeit"}}},
         "circulation": {"kind": "stairs"}},
        {"id": "B", "name": "Living and work",
         "geometry": {"relative_length": 1, "depth_m": 20, "max_floors": 6},
         "allocation": {"usages": ["office", "residential"]},
         "circulation": {"kind": "elevator"}},
    ],
}
section = section_layout_from_scenario(scenario, layout=layout)
show_section(render_svg(section))

Basements use an integer `max_basements` and fill from ground down. Without an explicit
list, eligible retail and school uses precede `other`; office and residential require
an explicit `basement_usages` entry. Remaining underground space is shown as schematic
parking and excluded from NFA. Above-ground usage areas are conserved as well.

`suggest_section_layout` provides an editable starting point if only the scenario is
known. The grouped version-1 form is JSON-compatible and suitable for a later HTML
parameter editor. Defaults merge into building overrides; lists replace inherited
lists, and an empty list clears them.

In [ ]:
suggested = suggest_section_layout(scenario, versioned=True)
display(pd.DataFrame([
    {"id": b["id"], **b["geometry"], "usages": b["allocation"]["usages"]}
    for b in suggested["buildings"]
]))
layout_json = json.dumps(normalize_layout(layout), ensure_ascii=False, indent=2)

## 4. Choose a drawing style

All styles use the same section. Architectural drawings use pastel usage fills,
construction and insulation bands, and restrained site silhouettes. The hand-drawn
style uses marker strokes, irregular frames and note labels. Blueprint uses a drafting
grid and a limited palette. Sky, paper and soil have subtle gradients.

Usage names come from the schema and appear inside spaces where they fit. The legend
retains the full names. Set `usage_labels="none"` to omit the internal labels.

In [ ]:
style_svgs = {}
for style in ["architectural", "hand_drawn", "blueprint"]:
    style_svgs[style] = render_svg(section, style=style)
    display(HTML(f"<h3>{style.replace('_', ' ').title()}</h3>"))
    show_section(style_svgs[style])

Blueprint accepts `palette="blue"`, `"green"` or `"beige"`. This changes drafting
colours while keeping heat, cold and airflow distinguishable. The default is blue.
`construction="wood"` selects a light-brown schematic construction layer in the
architectural style; neither choice changes material quantities or U-values.

In [ ]:
palette_svgs = {}
for palette in ["green", "beige"]:
    palette_svgs[palette] = render_svg(section, style="blueprint", palette=palette)
    display(HTML(f"<h3>Blueprint · {palette}</h3>"))
    show_section(palette_svgs[palette])

## 5. Draw building services where they act

`systems.usages` sets heating, cooling, ventilation and domestic hot water by usage.
The `"*"` entry is a shared default, with usage-specific values overriding it. Put
settings in `defaults.systems` to use them in every building, then override individual
buildings where needed. These are illustrative choices; editing them does not alter
the scenario's energy inputs.

TABS circuits sit within the structural slab, underfloor pipes above it, ceiling
panels below the ceiling, and radiators or split units beside façades. Ventilation
ducts follow the ceiling. The service shaft remains a drawing overlay without taking
area from a usage.

In [ ]:
services_layout = deepcopy(layout)
services_layout["defaults"]["systems"]["usages"] = {
    "*": {"heating": "underfloor", "ventilation": "balanced_recovery"},
    "office": {"heating": "tabs", "cooling": "ceiling_panel"},
    "retailfood": {"heating": "none", "cooling": "split", "ventilation": "exhaust"},
}
services_layout["buildings"][1]["systems"] = {
    "usages": {"residential": {"heating": "lt_radiator", "ventilation": "natural", "dhw": "electric_local"}},
}
services = section_layout_from_scenario(scenario, layout=services_layout)

# Annotation callouts follow any style. Try "architectural" or "hand_drawn" here.
annotated_svg = render_svg(services, style="blueprint", palette="beige", annotations=True)
show_section(annotated_svg)

### Preset examples in the same section

`preset_visuals()` contains all 40 entries from the recorded PEExcel preset catalogue:
heating, cooling, emission, ventilation and hot water. Each row retains its original
name, source range and cached values, alongside a runnable visual recipe and its
assumptions. The recipe illustrates the component; it does not apply a physical preset
to the scenario. Unknown heat sources and unspecified room terminals remain unresolved.

Five representative cases follow. The controls are ordinary cell values, so any
catalogue entry can be selected without a separate component drawing tool.

In [ ]:
catalog = preset_visuals()
display(pd.DataFrame(catalog)[["category", "name", "source"]])

selected_cases = [
    ("Heating", "Wärmepumpe (Grundwasser)"),
    ("Cooling", "WP reversibel - FreeCooling Anteil"),
    ("Emission", "Kühlpaneele"),
    ("Ventilation", "ABL-Anlage mit WRG"),
    ("Hot water", "zentrale WP"),
]
preset_svgs = {}
for category, name in selected_cases:
    case = next(row for row in catalog if (row["category"], row["name"]) == (category, name))
    case_layout = deepcopy(layout)
    # Demonstrate one local system in building B; building A remains a reference.
    case_layout["buildings"][1]["systems"] = deepcopy(case["systems"])
    case_section = section_layout_from_scenario(scenario, layout=case_layout)
    case_svg = render_svg(case_section, annotations=True)
    preset_svgs[category] = case_svg
    display(HTML(f"<h3>{category} · {name}</h3>"))
    print(case["notes"])
    show_section(case_svg)

To inspect another entry, change the category and name below. For example, compare
`ABL-Anlage mit WRG` with `Mech. Lüftung mit WRG`: heat recovery in an exhaust system
does not imply a second mechanical supply duct. Likewise, an `Erdwärme` preset does
not establish borehole geometry, and a reversible heat pump does not identify its
room emitter.

In [ ]:
category = "Emission"
preset_name = "Bauteilaktivierung"
case = next(row for row in preset_visuals(category) if row["name"] == preset_name)
case_layout = deepcopy(layout)
case_layout["buildings"][1]["systems"] = deepcopy(case["systems"])
chosen_section = section_layout_from_scenario(scenario, layout=case_layout)
print(case["notes"])
show_section(render_svg(chosen_section, style="hand_drawn", annotations=True))

## 6. Locate shared plant and add roof uses

Root-level `systems.plant` records describe one physical symbol with an explicit
host and `serves` references. A shared heat pump is therefore drawn once. By contrast,
plant in `defaults.systems` means a local instance in each building. Unknown root
locations remain unplaced. `level=0` denotes ground floor, negative levels basements;
`"lowest_floor"` and `"roof"` are convenient alternatives.

Roof options are PV, solar thermal, pergola, terrace, garden and an HVAC room continuing
the service shaft. Their modular scenes are schematic, not a roof-area optimisation.
PV retains the scenario-wide module-area budget and reports unplaced capacity.

In [ ]:
shared_layout = deepcopy(services_layout)
shared_layout["systems"] = {"plant": [{
    "id": "shared-heat-pump", "kind": "heat_pump", "source": "groundwater",
    "location": {"building": "A", "level": "lowest_floor"}, "serves": ["A", "B"],
}]}
shared_layout["buildings"][0]["systems"] = {"roof": ["pv", "solar_thermal", "hvac_room"]}
shared_layout["buildings"][1]["systems"]["roof"] = ["pv", "garden", "pergola", "terrace"]
shared_section = section_layout_from_scenario(scenario, layout=shared_layout)
shared_svg = render_svg(shared_section, annotations=True)
show_section(shared_svg)

## 7. Change PV inputs and compare real scenarios

Scientific parameters are changed through `scenario.v`. If `PV_module_area` is set,
it takes precedence over `PV_kWp × PV_m2_per_kWp`; clear it when deriving area from
capacity. Regenerate the section after editing inputs. `reuse_geometry` verifies
that area allocation and geometry are still compatible, preserving the comparison.

In [ ]:
v.PV_kWp = 180.0
v.PV_module_area = None
more_pv = section_layout_from_scenario(scenario, layout=layout, reuse_geometry=section)
more_pv_svg = render_svg(more_pv)
show_section(more_pv_svg)
display(pd.DataFrame([
    {"case": name, "requested_m2": s["pv"]["requested_area_m2"], "unplaced_m2": s["pv"]["unplaced_area_m2"]}
    for name, s in [("120 kWp", section), ("180 kWp", more_pv)]
]))
v.PV_kWp = 120.0

The imported Forsthausgasse variants can use the same layout too. The second variant
contains 185 m² of PV modules and 37 kWp in this export. Its name includes `WP`, but
the active stored heating capacity still indicates district heating. The drawing
follows populated inputs; a scenario name alone is not evidence for a heat pump.

In [ ]:
variant = next(s for s in project.scenarios if s.column_name == "Forsthausgasse | BT2 ÖKO WP FLEX 2,6xPV")
real_variant = section_layout_from_scenario(variant, layout=real_layout, reuse_geometry=real_section)
real_variant_svg = render_svg(real_variant, style="blueprint", palette="green", annotations=True)
show_section(real_variant_svg)
display(pd.DataFrame({s.column_name: {name: getattr(s.v, name, None) for name in fields}
                      for s in [baseline, variant]}))

## 8. Save figures and layout settings

SVG retains editable vector geometry, labels and semantic element IDs. Save the layout
alongside the drawing when a comparison needs to be reproduced. A later browser UI can
use those IDs for selection; this notebook deliberately uses static figures and cell
parameters. Output files below stay in the notebook's ignored `local_exports` folder.

In [ ]:
export_dir = ROOT / "notebooks/local_exports/visualize_project"
export_dir.mkdir(parents=True, exist_ok=True)
exports = {**style_svgs, **{f"blueprint_{p}": svg for p, svg in palette_svgs.items()},
           "services_annotated": annotated_svg, "shared_plant": shared_svg,
           "forsthausgasse": real_svg, "forsthausgasse_pv": real_variant_svg}
for name, svg in exports.items():
    (export_dir / f"{name}.svg").write_text(svg, encoding="utf-8")
(export_dir / "layout.json").write_text(layout_json, encoding="utf-8")
print(f"Saved {len(exports)} SVG figures and the layout to {export_dir}")

## Technical notes

- Areas are NFA. Space width times building depth equals its allocated area; drawing
  height is not part of this calculation. Horizontal and vertical scales differ.
- Layers, shaft widths, room partitions, streets, trees and plant positions are
  schematic. Fenestration follows the scenario ratio, or its wall/window areas when
  available; unknown geometry and defaults are recorded in `assumptions`.
- Above-ground usage budgets are conserved. Empty basement parking is outside those
  budgets. A partial top floor is preferable to inventing additional floor area.
- Symbol choices indicate topology and placement, not equipment capacity, circuit
  sizing or simulated performance. Preset recipes are explicit teaching examples.
- The hand-drawn font uses locally available Ink Free / Segoe Print / cursive fallbacks.
  Font appearance may differ between computers; no font service is required.
- Generation is deterministic. Reusing incompatible geometry raises an error; a seed
  alone does not guarantee stable layout after area changes.

The checks below cover area conservation, reproducibility, SVG parsing and all 40
catalogue recipes through the same renderer. The [API notes](../docs/development/building_sections.md)
describe the available fields, component names and module responsibilities.

In [ ]:
areas = pd.DataFrame(section["spaces"]).groupby("usage")["area_m2"].sum()
expected = pd.Series({name.removeprefix("NFA_"): area for name, area in usage_areas.items()})
area_check = pd.concat([expected.rename("input_m2"), areas.rename("section_m2")], axis=1)
area_check["difference_m2"] = area_check["section_m2"] - area_check["input_m2"]
assert area_check["difference_m2"].abs().max() < 1e-7
assert section == section_layout_from_scenario(scenario, layout=layout)
assert normalize_layout(json.loads(layout_json)) == normalize_layout(layout)
for svg in exports.values():
    ET.fromstring(svg)

snapshot = deepcopy(scenario.as_dict())
for case in catalog:
    check_layout = deepcopy(layout)
    check_layout["buildings"][1]["systems"] = deepcopy(case["systems"])
    check_section = section_layout_from_scenario(scenario, layout=check_layout)
    assert check_section["geometry_signature"] == section["geometry_signature"]
    ET.fromstring(render_svg(check_section, annotations=True))
assert scenario.as_dict() == snapshot
display(area_check)
print(f"Validated {len(catalog)} catalogue recipes, conserved areas and unchanged scenario inputs.")

In [ ]:
display(pd.DataFrame(section["assumptions"]))